# OmniDoc-RAG: Stage 1 Contrastive Pretraining
### OCR-Free Multi-Modal Document Retrieval on Kaggle Free GPUs (2x NVIDIA T4)

This notebook trains the **Custom 2D-RoPE Projection Layer** and **Perceiver Resampler Bottleneck** end-to-end using **Symmetric Patch-InfoNCE Contrastive Loss with Late Interaction (MaxSim)** on real DocVQA document page images.

In [ ]:
# Step 1: Install core dependencies in Kaggle environment
!pip install -q pymupdf einops datasets accelerate wandb

import os
import math
import time
from typing import Tuple, Dict, Any, Optional, List
from PIL import Image
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from einops import rearrange, repeat

# Check CUDA GPU availability
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Mathematical Architecture Modules
* **`RotaryEmbedding2D`**: 2D Spatial Rotary Position Embedding Layer (analytic relative displacement invariance $\Delta y, \Delta x$).
* **`PerceiverResampler`**: Learned latent cross-attention bottleneck ($1024 \to 64$ tokens).
* **`SymmetricPatchInfoNCELoss`**: Multi-Vector Late-Interaction (MaxSim) Symmetric Contrastive Objective.

In [ ]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    half_dim = x.shape[-1] // 2
    x1, x2 = x[..., :half_dim], x[..., half_dim:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_emb_2d(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    return (x * cos) + (rotate_half(x) * sin)

class RotaryEmbedding2D(nn.Module):
    def __init__(self, dim: int, base: float = 10000.0, max_height: int = 64, max_width: int = 64):
        super().__init__()
        self.dim = dim
        self.dim_axis = dim // 2
        inv_freq = 1.0 / (base ** (torch.arange(0, self.dim_axis, 2).float() / self.dim_axis))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self._cached_h, self._cached_w = 0, 0
        self._cached_cos, self._cached_sin = None, None

    def get_cos_sin(self, height: int, width: int, device: torch.device, dtype: torch.dtype = torch.float32):
        if (self._cached_cos is None or self._cached_h != height or self._cached_w != width or
            self._cached_cos.device != device or self._cached_cos.dtype != dtype):
            inv_freq = self.inv_freq.to(device=device, dtype=dtype)
            y_pos = torch.arange(height, device=device, dtype=dtype)
            x_pos = torch.arange(width, device=device, dtype=dtype)
            freqs_y = torch.outer(y_pos, inv_freq).view(height, 1, -1).expand(height, width, -1)
            freqs_x = torch.outer(x_pos, inv_freq).view(1, width, -1).expand(height, width, -1)
            freqs_2d = torch.cat([freqs_y, freqs_x], dim=-1).view(height * width, -1)
            emb = torch.cat([freqs_2d, freqs_2d], dim=-1)
            self._cached_h, self._cached_w = height, width
            self._cached_cos = emb.cos().view(1, 1, height * width, self.dim)
            self._cached_sin = emb.sin().view(1, 1, height * width, self.dim)
        return self._cached_cos, self._cached_sin

    def forward(self, q: torch.Tensor, k: torch.Tensor, height: int, width: int):
        cos, sin = self.get_cos_sin(height, width, device=q.device, dtype=q.dtype)
        return apply_rotary_emb_2d(q, cos, sin), apply_rotary_emb_2d(k, cos, sin)

class FeedForward(nn.Module):
    def __init__(self, dim: int, mult: int = 4, dropout: float = 0.0):
        super().__init__()
        inner = int(dim * mult)
        self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, inner),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(inner, dim),
            nn.Dropout(dropout)
        )
    def forward(self, x): return self.net(x)

class PerceiverAttention(nn.Module):
    def __init__(self, dim: int, heads: int = 8, head_dim: int = 64, dropout: float = 0.0):
        super().__init__()
        self.heads, self.head_dim = heads, head_dim
        inner_dim = heads * head_dim
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.to_q = nn.Linear(dim, inner_dim, bias=False)
        self.to_k = nn.Linear(dim, inner_dim, bias=False)
        self.to_v = nn.Linear(dim, inner_dim, bias=False)
        self.to_out = nn.Sequential(nn.Linear(inner_dim, dim, bias=False), nn.Dropout(dropout))

    def forward(self, q_in, kv_in, rope_2d=None, grid_hw=None):
        q = rearrange(self.to_q(self.norm_q(q_in)), "b k (h d) -> b h k d", h=self.heads)
        k = rearrange(self.to_k(self.norm_kv(kv_in)), "b n (h d) -> b h n d", h=self.heads)
        v = rearrange(self.to_v(self.norm_kv(kv_in)), "b n (h d) -> b h n d", h=self.heads)
        if rope_2d is not None and grid_hw is not None:
            cos, sin = rope_2d.get_cos_sin(grid_hw[0], grid_hw[1], device=k.device, dtype=k.dtype)
            k = apply_rotary_emb_2d(k, cos, sin)
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0 if not self.training else 0.05)
        return self.to_out(rearrange(out, "b h k d -> b k (h d)"))

class PerceiverBlock(nn.Module):
    def __init__(self, dim: int, heads: int = 8, head_dim: int = 64, ff_mult: int = 4, dropout: float = 0.0):
        super().__init__()
        self.cross_attn = PerceiverAttention(dim, heads=heads, head_dim=head_dim, dropout=dropout)
        self.self_attn = PerceiverAttention(dim, heads=heads, head_dim=head_dim, dropout=dropout)
        self.ffn = FeedForward(dim, mult=ff_mult, dropout=dropout)
    def forward(self, latents, x, rope_2d=None, grid_hw=None):
        latents = latents + self.cross_attn(latents, x, rope_2d=rope_2d, grid_hw=grid_hw)
        latents = latents + self.self_attn(latents, latents)
        return latents + self.ffn(latents)

class PerceiverResampler(nn.Module):
    def __init__(self, dim: int = 512, depth: int = 2, num_latents: int = 64, heads: int = 8, head_dim: int = 64, use_rope2d: bool = True):
        super().__init__()
        self.dim = dim
        self.latents = nn.Parameter(torch.randn(num_latents, dim) * 0.02)
        self.rope_2d = RotaryEmbedding2D(dim=head_dim) if use_rope2d else None
        self.layers = nn.ModuleList([PerceiverBlock(dim=dim, heads=heads, head_dim=head_dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor, grid_hw: Optional[Tuple[int, int]] = None):
        b, n, d = x.shape
        if grid_hw is None and self.rope_2d is not None:
            side = int(math.isqrt(n))
            if side * side == n: grid_hw = (side, side)
        latents = repeat(self.latents, "k d -> b k d", b=b)
        for layer in self.layers:
            latents = layer(latents, x, rope_2d=self.rope_2d, grid_hw=grid_hw)
        return self.norm(latents)

class SymmetricPatchInfoNCELoss(nn.Module):
    def __init__(self, init_temperature: float = 0.07, min_temperature: float = 0.01, max_temperature: float = 1.0):
        super().__init__()
        self.min_temp, self.max_temp = min_temperature, max_temperature
        self.log_tau = nn.Parameter(torch.tensor(math.log(init_temperature), dtype=torch.float32))

    @property
    def temperature(self):
        return torch.clamp(torch.exp(self.log_tau), min=self.min_temp, max=self.max_temp)

    def compute_maxsim_scores(self, queries: torch.Tensor, documents: torch.Tensor, query_mask: Optional[torch.Tensor] = None):
        q_norm = F.normalize(queries, p=2, dim=-1)
        d_norm = F.normalize(documents, p=2, dim=-1)
        sim_matrix = torch.einsum("b l d, c k d -> b c l k", q_norm, d_norm)
        max_sim = sim_matrix.max(dim=-1).values  # (B_q, B_d, L)
        if query_mask is not None:
            max_sim = max_sim * query_mask.unsqueeze(1).to(dtype=max_sim.dtype)
        return max_sim.sum(dim=-1)  # (B_q, B_d)

    def forward(self, queries: torch.Tensor, documents: torch.Tensor, query_mask: Optional[torch.Tensor] = None):
        b = queries.shape[0]
        raw_scores = self.compute_maxsim_scores(queries, documents, query_mask=query_mask)
        tau = self.temperature
        scaled_scores = raw_scores / tau
        targets = torch.arange(b, device=queries.device, dtype=torch.long)
        loss_q2d = F.cross_entropy(scaled_scores, targets)
        loss_d2q = F.cross_entropy(scaled_scores.transpose(0, 1), targets)
        total_loss = 0.5 * (loss_q2d + loss_d2q)
        metrics = {"loss": total_loss.detach(), "loss_q2d": loss_q2d.detach(), "loss_d2q": loss_d2q.detach(), "temperature": tau.detach()}
        return total_loss, metrics

class VisionPatchExtractor(nn.Module):
    def __init__(self, in_channels: int = 3, embed_dim: int = 512, patch_size: int = 32):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, x):
        feat = self.proj(x)
        h_grid, w_grid = feat.shape[2], feat.shape[3]
        patches = self.norm(rearrange(feat, "b d h w -> b (h w) d"))
        return patches, (h_grid, w_grid)

class QueryEmbedding(nn.Module):
    def __init__(self, vocab_size: int = 30522, embed_dim: int = 512, max_seq_len: int = 128):
        super().__init__()
        self.tokens = nn.Embedding(vocab_size, embed_dim)
        self.pos = nn.Embedding(max_seq_len, embed_dim)
        self.norm = nn.LayerNorm(embed_dim)
    def forward(self, input_ids, attention_mask=None):
        positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
        return self.norm(self.tokens(input_ids) + self.pos(positions))

class OmniDocDualEncoder(nn.Module):
    def __init__(self, embed_dim: int = 512, patch_size: int = 32, num_latents: int = 64, perceiver_depth: int = 2, heads: int = 8, vocab_size: int = 30522):
        super().__init__()
        head_dim = embed_dim // heads
        self.patch_extractor = VisionPatchExtractor(3, embed_dim, patch_size)
        self.perceiver = PerceiverResampler(embed_dim, perceiver_depth, num_latents, heads, head_dim, use_rope2d=True)
        self.query_encoder = QueryEmbedding(vocab_size, embed_dim)
        self.loss_fn = SymmetricPatchInfoNCELoss(init_temperature=0.07)

    def encode_document(self, images: torch.Tensor) -> torch.Tensor:
        patches, grid_hw = self.patch_extractor(images)
        return self.perceiver(patches, grid_hw=grid_hw)

    def encode_query(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        return self.query_encoder(input_ids, attention_mask=attention_mask)

    def forward(self, images, input_ids, attention_mask=None):
        d = self.encode_document(images)
        q = self.encode_query(input_ids, attention_mask=attention_mask)
        return self.loss_fn(q, d, query_mask=attention_mask)

print("✓ OmniDoc-RAG Neural Engine Loaded Successfully!")

## 2. Real Document Data Pipeline (Robust Multi-Lingual & String Query Parsing)
Loads real scanned document pages and natural language queries.

In [ ]:
from datasets import load_dataset

class DocVQADataset(Dataset):
    IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __init__(self, hf_data, target_size=(1024, 1024), max_samples=None):
        if max_samples is not None and max_samples < len(hf_data):
            self.samples = [hf_data[i] for i in range(max_samples)]
        else:
            self.samples = list(hf_data)
        self.target_size = target_size

    def __len__(self):
        return len(self.samples)

    def _preprocess_image(self, img: Image.Image) -> torch.Tensor:
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        if img.mode != "RGB":
            img = img.convert("RGB")
        orig_w, orig_h = img.size
        scale = min(self.target_size[0] / orig_w, self.target_size[1] / orig_h)
        new_w, new_h = max(1, int(orig_w * scale)), max(1, int(orig_h * scale))
        resized = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        canvas = Image.new("RGB", self.target_size, (255, 255, 255))
        canvas.paste(resized, (0, 0))
        arr = np.array(canvas, dtype=np.float32) / 255.0
        tensor = torch.from_numpy(arr).permute(2, 0, 1)
        return (tensor - self.IMAGENET_MEAN) / self.IMAGENET_STD

    def __getitem__(self, idx):
        item = self.samples[idx]
        img_tensor = self._preprocess_image(item["image"])
        
        # Robust query parsing: handles strings, dicts ({'en': '...'}), and lists
        raw_query = item.get("query", item.get("question", "What is shown in this document?"))
        if isinstance(raw_query, dict):
            query = str(raw_query.get("en", next(iter(raw_query.values()), "")))
        elif isinstance(raw_query, (list, tuple)):
            query = str(raw_query[0]) if len(raw_query) > 0 else ""
        else:
            query = str(raw_query)
            
        doc_id = str(item.get("id", item.get("questionId", idx)))
        return {
            "image": img_tensor,
            "question": query,
            "doc_id": doc_id
        }

class OmniDocCollate:
    def __init__(self, max_query_len=48):
        self.max_query_len = max_query_len

    def __call__(self, batch):
        images = torch.stack([item["image"] for item in batch], dim=0)
        questions = [str(item["question"]) for item in batch]
        max_len = min(self.max_query_len, max(len(q.split()) for q in questions) + 2)
        b = len(questions)
        input_ids = torch.zeros((b, max_len), dtype=torch.long)
        attention_mask = torch.zeros((b, max_len), dtype=torch.float32)
        for i, q in enumerate(questions):
            tokens = [abs(hash(w)) % 30000 + 1 for w in q.split()][:max_len - 1]
            input_ids[i, 0] = 101
            input_ids[i, 1:len(tokens)+1] = torch.tensor(tokens, dtype=torch.long)
            attention_mask[i, :len(tokens)+1] = 1.0
        return {"images": images, "input_ids": input_ids, "attention_mask": attention_mask, "questions": questions}

# Load verified public DocVQA dataset from Hugging Face
print("Loading verified DocVQA dataset from Hugging Face...")
try:
    raw_data = load_dataset("nielsr/docvqa_1200_examples", split="train")
    train_dataset = DocVQADataset(raw_data, target_size=(1024, 1024))
    print(f"✓ Successfully loaded {len(train_dataset)} real DocVQA document images & questions!")
except Exception as e:
    print(f"Trying alternative mirror: {e}")
    raw_data = load_dataset("lmms-lab/DocVQA", "DocVQA", split="validation")
    train_dataset = DocVQADataset(raw_data, target_size=(1024, 1024), max_samples=2000)
    print(f"✓ Loaded {len(train_dataset)} DocVQA samples from mirror!")

## 3. Mixed-Precision (FP16) Training Loop with Cosine Warmup & GradScaler

In [ ]:
# Training Hyperparameters
BATCH_SIZE = 8
EPOCHS = 3
PEAK_LR = 1e-4
EMBED_DIM = 512
NUM_LATENTS = 64
GRAD_ACCUM_STEPS = 2

collate_fn = OmniDocCollate(max_query_len=48)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)

model = OmniDocDualEncoder(
    embed_dim=EMBED_DIM,
    patch_size=32,
    num_latents=NUM_LATENTS,
    perceiver_depth=2,
    heads=8
).to(device)

optimizer = AdamW(model.parameters(), lr=PEAK_LR, weight_decay=0.01)
total_training_steps = (len(train_loader) // GRAD_ACCUM_STEPS) * EPOCHS

def lr_lambda(step):
    warmup = 50
    if step < warmup: return float(step) / float(max(1, warmup))
    progress = float(step - warmup) / float(max(1, total_training_steps - warmup))
    return max(0.01, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

print(f"\n[OmniDoc-RAG] Training on {device} | Total Batches/Epoch: {len(train_loader)} | Total Steps: {total_training_steps}")
model.train()
global_step = 0

for epoch in range(EPOCHS):
    epoch_loss = 0.0
    start_epoch = time.time()
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        images = batch["images"].to(device, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        mask = batch["attention_mask"].to(device, non_blocking=True)

        # Mixed-Precision Forward Pass
        if torch.cuda.is_available():
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss, metrics = model(images, input_ids, attention_mask=mask)
                loss = loss / GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
        else:
            loss, metrics = model(images, input_ids, attention_mask=mask)
            loss = loss / GRAD_ACCUM_STEPS
            loss.backward()

        epoch_loss += loss.item() * GRAD_ACCUM_STEPS

        # Gradient Accumulation Step
        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            if torch.cuda.is_available():
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            optimizer.zero_grad()
            scheduler.step()
            global_step += 1

            if global_step % 10 == 0 or global_step == 1:
                print(f"Epoch [{epoch+1}/{EPOCHS}] | Step [{global_step}/{total_training_steps}] | Loss: {metrics['loss']:.4f} | Q->D: {metrics['loss_q2d']:.4f} | D->Q: {metrics['loss_d2q']:.4f} | Tau: {metrics['temperature']:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    print(f"Epoch {epoch+1} finished in {time.time() - start_epoch:.2f}s | Avg Loss: {epoch_loss / len(train_loader):.4f}")

# Save trained checkpoint
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/omnidoc_stage1_best.pt")
print("\n✓ Trained Model Checkpoint saved to checkpoints/omnidoc_stage1_best.pt!")

## 4. Evaluation Suite: Retrieval Recall@1, Recall@5 & MRR

In [ ]:
@torch.no_grad()
def evaluate_retrieval_benchmarks(model, eval_loader, device):
    model.eval()
    all_doc_latents = []
    all_query_batches = []
    all_mask_batches = []

    print("Encoding real document pages and text queries for evaluation...")
    for batch in eval_loader:
        images = batch["images"].to(device)
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        doc_latents = model.encode_document(images)
        query_embeds = model.encode_query(input_ids, attention_mask=mask)

        all_doc_latents.append(doc_latents.cpu())
        all_query_batches.append(query_embeds.cpu())
        all_mask_batches.append(mask.cpu())

    all_doc_latents = torch.cat(all_doc_latents, dim=0).to(device)  # (N_docs, K, D)
    n_docs = len(all_doc_latents)
    d_norm = F.normalize(all_doc_latents, p=2, dim=-1)

    print(f"Evaluating MaxSim retrieval across {n_docs} real document-query pairs...")
    all_scores = []
    for q_batch, m_batch in zip(all_query_batches, all_mask_batches):
        q_norm = F.normalize(q_batch.to(device), p=2, dim=-1)
        mask = m_batch.to(device)
        # Compute pairwise MaxSim score between batch of queries and all documents
        sim = torch.einsum("b l d, c k d -> b c l k", q_norm, d_norm)
        max_sim = sim.max(dim=-1).values  # (B_q, N_docs, L)
        if mask is not None:
            max_sim = max_sim * mask.unsqueeze(1)
        score_b = max_sim.sum(dim=-1)     # (B_q, N_docs)
        all_scores.append(score_b.cpu())

    scores = torch.cat(all_scores, dim=0).to(device)  # (N_queries, N_docs)
    targets = torch.arange(n_docs, device=device).unsqueeze(1)
    ranked_indices = torch.argsort(scores, dim=-1, descending=True)

    r1 = (ranked_indices[:, :1] == targets).any(dim=-1).float().mean().item() * 100
    r5 = (ranked_indices[:, :5] == targets).any(dim=-1).float().mean().item() * 100
    ranks = (ranked_indices == targets).nonzero()[:, 1] + 1
    mrr = (1.0 / ranks.float()).mean().item()

    print(f"\n================ Retrieval Results on Real Documents ================")
    print(f"Recall@1: {r1:.2f}%")
    print(f"Recall@5: {r5:.2f}%")
    print(f"MRR (Mean Reciprocal Rank): {mrr:.4f}")
    print(f"=====================================================================")
    return {"Recall@1": r1, "Recall@5": r5, "MRR": mrr}

# Run evaluation across real dataset
eval_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
metrics = evaluate_retrieval_benchmarks(model, eval_loader, device)